# AutoGen Debate — Two-Agent Turn-Based Argumentation
## For vs. Against — Two-Agent Turn-Based Debate
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/21-autogen-debate/autogen_debate_workbook.ipynb)

Two AutoGen ConversableAgents debate a topic: a Proponent argues FOR and an Opponent argues AGAINST. AutoGen manages the turn loop via initiate_chat — contrast to LangGraph's explicit StateGraph.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — AutoGen vs LangGraph — internal turn loop vs explicit graph |
| 2 | **ConversableAgent** — System prompt per role; human_input_mode="NEVER"; llm_config |
| 3 | **initiate_chat()** — max_turns=4; silent=True; returns chat_history |
| 4 | **Debate Analysis** — Parse turns, count arguments, extract positions |
| 5 | **Framework Contrast** — Same debate in LangGraph for comparison |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "pyautogen", "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
from autogen import ConversableAgent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from typing import TypedDict
import os

MAX_TURNS = 4
DEBATE_TOPICS = [
    "Remote work is more productive than in-office work",
    "AI will create more jobs than it eliminates",
]

llm_config = {"model": "gpt-4o-mini", "api_key": os.environ.get("OPENAI_API_KEY")}

def run_debate(topic: str) -> list:
    proponent = ConversableAgent(
        name="Proponent",
        system_message=f"You are a skilled debater arguing FOR: '{topic}'. Be concise — 2-3 sentences per turn.",
        llm_config=llm_config, human_input_mode="NEVER",
    )
    opponent = ConversableAgent(
        name="Opponent",
        system_message=f"You are a skilled debater arguing AGAINST: '{topic}'. Counter each argument. 2-3 sentences per turn.",
        llm_config=llm_config, human_input_mode="NEVER",
    )
    result = proponent.initiate_chat(
        opponent,
        message=f"Proposition: '{topic}'. I argue FOR it.",
        max_turns=MAX_TURNS, silent=True,
    )
    return result.chat_history

topic = DEBATE_TOPICS[0]
print(f"Topic: {topic}")
history = run_debate(topic)
print(f"Turns: {len(history)}")
for i, msg in enumerate(history):
    speaker = msg.get("name", msg.get("role", "?"))
    print(f"  Turn {i+1} [{speaker}]: {msg['content'][:120]}...")

print()
print("Framework comparison:")
print("  AutoGen: ConversableAgent + initiate_chat() — turn loop managed by framework")
print("  LangGraph: StateGraph + explicit nodes/edges — full control over each step")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*